In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root / "src"))

# Optimizer experiments

Verify the TinyTorch `SGD` and `Adam` optimizers in four passes, in this order:

1. **Manual correctness** вЂ” hand-computed formulas for every update rule.
2. **PyTorch parity** вЂ” identical initial parameters and manually assigned gradients, compared against `torch.optim` step by step.
3. **Integration training** вЂ” real `backward()` + `step()` + `zero_grad()` loops on a quadratic and a linear regression.
4. **Performance** вЂ” milliseconds per `step()`, optimizer-state memory, and memory stability across repeated steps.

Both optimizers update `Tensor_CP.data` in place from the NumPy gradient stored in `.grad`.

In [3]:
from statistics import mean, median
from time import perf_counter

import numpy as np
import psutil
import torch

from tinytorch.loss_functions import MSELoss_CP
from tinytorch.optimizers import SGD, Adam
from tinytorch.tensor import Tensor_CP

np.random.seed(42)
torch.manual_seed(42)

ATOL, RTOL = 1e-6, 1e-5

print(f"NumPy:   {np.__version__}")
print(f"PyTorch: {torch.__version__}")
print("Device:  CPU")

NumPy:   2.5.0
PyTorch: 2.12.1+cpu
Device:  CPU


## 1. Shared optimizer tests

### Constructor validation

Both optimizers must reject malformed inputs: non-list `params`, non-`Tensor_CP` elements, `lr <= 0`, negative `weight_decay`, momentum outside [0, 1], betas outside [0, 1), and `epsilon <= 0`.

Three behaviors of the current implementation are *documented* rather than enforced (see the next cell): an empty parameter list is deliberately allowed as a no-op, duplicate tensors in `params` update the parameter twice per step, and booleans such as `lr=True` are accepted because `bool` is an `int` subclass in Python.

In [4]:
def expect_raises(exception_type, callback, description):
    try:
        callback()
    except exception_type:
        print(f"OK   {description}")
        return
    except Exception as error:
        raise AssertionError(
            f"{description}: expected {exception_type.__name__}, "
            f"got {type(error).__name__}: {error}"
        )
    raise AssertionError(
        f"{description}: expected {exception_type.__name__}, nothing was raised"
    )


def make_param(value=(1.0, -2.0)):
    return Tensor_CP(np.array(value, dtype=np.float32), requires_grad=True)


for optimizer_class in (SGD, Adam):
    name = optimizer_class.__name__
    expect_raises(TypeError, lambda cls=optimizer_class: cls(params=(make_param(),)),
                  f"{name} rejects non-list params")
    expect_raises(TypeError, lambda cls=optimizer_class: cls(params=[make_param(), 3.0]),
                  f"{name} rejects a non-Tensor_CP element")
    expect_raises(ValueError, lambda cls=optimizer_class: cls(params=[make_param()], lr=0.0),
                  f"{name} rejects lr = 0")
    expect_raises(ValueError, lambda cls=optimizer_class: cls(params=[make_param()], lr=-0.1),
                  f"{name} rejects lr < 0")
    expect_raises(ValueError, lambda cls=optimizer_class: cls(params=[make_param()], weight_decay=-0.01),
                  f"{name} rejects negative weight_decay")

expect_raises(ValueError, lambda: SGD([make_param()], momentum=-0.1), "SGD rejects momentum < 0")
expect_raises(ValueError, lambda: SGD([make_param()], momentum=1.5), "SGD rejects momentum > 1")

expect_raises(ValueError, lambda: Adam([make_param()], betas=(1.0, 0.999)), "Adam rejects beta1 = 1")
expect_raises(ValueError, lambda: Adam([make_param()], betas=(-0.1, 0.999)), "Adam rejects beta1 < 0")
expect_raises(ValueError, lambda: Adam([make_param()], betas=(0.9, 1.0)), "Adam rejects beta2 = 1")
expect_raises(ValueError, lambda: Adam([make_param()], betas=(0.9, -0.1)), "Adam rejects beta2 < 0")
expect_raises(ValueError, lambda: Adam([make_param()], epsilon=0.0), "Adam rejects epsilon = 0")
expect_raises(ValueError, lambda: Adam([make_param()], epsilon=-1e-8), "Adam rejects epsilon < 0")

print("\nConstructor validation passed for SGD and Adam")

OK   SGD rejects non-list params
OK   SGD rejects a non-Tensor_CP element
OK   SGD rejects lr = 0
OK   SGD rejects lr < 0
OK   SGD rejects negative weight_decay
OK   Adam rejects non-list params
OK   Adam rejects a non-Tensor_CP element
OK   Adam rejects lr = 0
OK   Adam rejects lr < 0
OK   Adam rejects negative weight_decay
OK   SGD rejects momentum < 0
OK   SGD rejects momentum > 1
OK   Adam rejects beta1 = 1
OK   Adam rejects beta1 < 0
OK   Adam rejects beta2 = 1
OK   Adam rejects beta2 < 0
OK   Adam rejects epsilon = 0
OK   Adam rejects epsilon < 0

Constructor validation passed for SGD and Adam


### Documented behavior вЂ” empty lists, duplicates, booleans

These three cases are accepted by the current implementation. The asserts pin down the actual behavior so any future change surfaces here:

- An **empty parameter list** constructs successfully and `step()` is a no-op (deliberately allowed).
- **`params=[x, x]`** applies the update twice per `step()` вЂ” a known limitation; rejecting duplicate tensor identities in `Optimizer.__init__()` would be the stronger design.
- **`lr=True`** is accepted and behaves as `lr=1.0` (`bool` is an `int` subclass) вЂ” a known limitation that applies to the other numeric hyperparameters as well.

In [5]:
# Empty parameter list: allowed, step() and zero_grad() are no-ops
for optimizer in (SGD([], lr=0.1), Adam([], lr=0.1)):
    optimizer.step()
    optimizer.zero_grad()
print("OK   empty parameter list is deliberately allowed as a no-op")

# Duplicate parameters: the tensor is updated once per occurrence
theta = Tensor_CP(np.array(1.0, dtype=np.float32), requires_grad=True)
theta.grad = np.array(0.5, dtype=np.float32)
SGD([theta, theta], lr=0.1).step()
print("theta after one step with params=[x, x]:", theta.data)
assert np.allclose(theta.data, 0.9, atol=ATOL), "expected two applications of -lr * grad"
print("OK   known limitation: duplicates update the parameter twice per step")

# Boolean lr: accepted, behaves as lr = 1.0
theta = Tensor_CP(np.array(1.0, dtype=np.float32), requires_grad=True)
theta.grad = np.array(0.5, dtype=np.float32)
SGD([theta], lr=True).step()
assert np.allclose(theta.data, 0.5, atol=ATOL), "lr=True behaves as lr=1.0"
print("OK   known limitation: lr=True is accepted and acts as lr=1.0")

OK   empty parameter list is deliberately allowed as a no-op
theta after one step with params=[x, x]: 0.9
OK   known limitation: duplicates update the parameter twice per step
OK   known limitation: lr=True is accepted and acts as lr=1.0


### Common behavior

For every optimizer configuration: `step()` returns `None`, a `grad=None` parameter is skipped, shapes and dtypes are preserved, `.grad` is never mutated by `step()`, parameters update independently, `zero_grad()` resets every gradient to `None`, and scalar/vector/matrix parameters all work.

Note the PyTorch distinction this suite keeps testing: `grad=None` **skips** a parameter entirely, while a gradient full of zeros **still participates** in optimizer logic (weight decay, buffer creation). The numerical consequences are covered in SGD test 5 and Adam test 4.

In [6]:
def clone_param(values):
    return Tensor_CP(np.array(values, dtype=np.float32), requires_grad=True)


for optimizer_factory, label in (
    (lambda ps: SGD(ps, lr=0.1), "SGD"),
    (lambda ps: SGD(ps, lr=0.1, momentum=0.9, weight_decay=0.01), "SGD(momentum, weight_decay)"),
    (lambda ps: Adam(ps, lr=0.1), "Adam"),
    (lambda ps: Adam(ps, lr=0.1, weight_decay=0.01), "Adam(weight_decay)"),
):
    scalar = clone_param(1.5)
    vector = clone_param([1.0, -2.0, 3.0])
    matrix = clone_param([[1.0, 2.0], [3.0, 4.0]])
    frozen = clone_param([5.0, 6.0])  # keeps grad=None the whole time

    optimizer = optimizer_factory([scalar, vector, matrix, frozen])

    scalar.grad = np.array(0.5, dtype=np.float32)
    vector.grad = np.array([0.1, -0.2, 0.3], dtype=np.float32)
    matrix.grad = np.array([[1.0, -1.0], [0.5, -0.5]], dtype=np.float32)

    named = [("scalar", scalar), ("vector", vector), ("matrix", matrix)]
    before = {name: parameter.data.copy() for name, parameter in named}
    frozen_before = frozen.data.copy()
    vector_grad_before = vector.grad.copy()

    result = optimizer.step()

    assert result is None, "step() must return None"
    assert np.array_equal(frozen.data, frozen_before), "grad=None parameter must stay unchanged"
    for name, parameter in named:
        assert parameter.data.shape == before[name].shape, f"{name}: shape changed"
        assert parameter.data.dtype == np.float32, f"{name}: dtype changed"
        assert not np.array_equal(parameter.data, before[name]), f"{name}: no update applied"
    assert np.array_equal(vector.grad, vector_grad_before), "step() must not modify tensor.grad"

    # Independence: only the vector gets a new gradient; the others must stay put.
    scalar_snapshot = scalar.data.copy()
    matrix_snapshot = matrix.data.copy()
    vector.grad = np.array([1.0, 1.0, 1.0], dtype=np.float32)
    scalar.grad = None
    matrix.grad = None
    optimizer.step()
    assert np.array_equal(scalar.data, scalar_snapshot), "scalar moved without a gradient"
    assert np.array_equal(matrix.data, matrix_snapshot), "matrix moved without a gradient"

    optimizer.zero_grad()
    assert all(parameter.grad is None for parameter in (scalar, vector, matrix, frozen)), \
        "zero_grad() must set every parameter gradient to None"
    print(f"OK   {label}")

print("\nCommon behavior passed for all configurations")

OK   SGD
OK   SGD(momentum, weight_decay)
OK   Adam
OK   Adam(weight_decay)

Common behavior passed for all configurations


## 2. SGD tests

### Test 1 вЂ” plain SGD, one step

With theta = [1, -2], grad = [0.5, -1], lr = 0.1 the update theta - lr * grad must give [0.95, -1.9], and `.grad` must stay untouched.

In [7]:
theta = Tensor_CP(np.array([1.0, -2.0], dtype=np.float32), requires_grad=True)
theta.grad = np.array([0.5, -1.0], dtype=np.float32)

SGD([theta], lr=0.1).step()

print("theta after one step:", theta.data)
assert np.allclose(theta.data, [0.95, -1.9], atol=ATOL, rtol=RTOL)
assert np.array_equal(theta.grad, np.array([0.5, -1.0], dtype=np.float32))
print("Plain SGD matches theta - lr * grad, .grad untouched")

theta after one step: [ 0.95 -1.9 ]
Plain SGD matches theta - lr * grad, .grad untouched


### Test 2 вЂ” weight decay only

Coupled weight decay first modifies the gradient: g' = g + lambda * theta = [0.51, -1.02], then theta - 0.1 * g' = [0.949, -1.898]. The stored `.grad` must remain the original [0.5, -1.0].

In [8]:
theta = Tensor_CP(np.array([1.0, -2.0], dtype=np.float32), requires_grad=True)
theta.grad = np.array([0.5, -1.0], dtype=np.float32)

SGD([theta], lr=0.1, weight_decay=0.01).step()

print("theta after one step:", theta.data)
assert np.allclose(theta.data, [0.949, -1.898], atol=ATOL, rtol=RTOL)
assert np.array_equal(theta.grad, np.array([0.5, -1.0], dtype=np.float32))
print("Weight decay applied to the update only; .grad untouched")

theta after one step: [ 0.949 -1.898]
Weight decay applied to the update only; .grad untouched


### Test 3 вЂ” momentum over two steps

Scalar theta_0 = 1, lr = 0.1, mu = 0.9. TinyTorch uses the same velocity convention as PyTorch: v_new = mu * v_old + g, with the learning rate applied only in the final parameter update.

- Step 1 (g = 2): v_1 = 2, theta_1 = 1 - 0.1 * 2 = 0.8
- Step 2 (g = 3): v_2 = 0.9 * 2 + 3 = 4.8, theta_2 = 0.8 - 0.48 = 0.32

In [9]:
theta = Tensor_CP(np.array(1.0, dtype=np.float32), requires_grad=True)
optimizer = SGD([theta], lr=0.1, momentum=0.9)

theta.grad = np.array(2.0, dtype=np.float32)
optimizer.step()
print("after step 1:", theta.data)
assert np.allclose(theta.data, 0.8, atol=ATOL, rtol=RTOL)

theta.grad = np.array(3.0, dtype=np.float32)
optimizer.step()
print("after step 2:", theta.data)
assert np.allclose(theta.data, 0.32, atol=ATOL, rtol=RTOL)
assert np.allclose(optimizer.momentum_buffers[id(theta)], 4.8, atol=ATOL, rtol=RTOL)
print("Momentum velocity buffer holds 4.8 as expected")

after step 1: 0.8
after step 2: 0.32
Momentum velocity buffer holds 4.8 as expected


### Test 4 вЂ” momentum and weight decay together

theta_0 = 1, lr = 0.1, mu = 0.9, lambda = 0.1.

- Step 1 (g = 2): g' = 2 + 0.1 * 1 = 2.1, v_1 = 2.1, theta_1 = 1 - 0.21 = 0.79
- Step 2 (g = 1): g' = 1 + 0.1 * 0.79 = 1.079, v_2 = 0.9 * 2.1 + 1.079 = 2.969, theta_2 = 0.79 - 0.2969 = 0.4931

In [10]:
theta = Tensor_CP(np.array(1.0, dtype=np.float32), requires_grad=True)
optimizer = SGD([theta], lr=0.1, momentum=0.9, weight_decay=0.1)

theta.grad = np.array(2.0, dtype=np.float32)
optimizer.step()
print("after step 1:", theta.data)
assert np.allclose(theta.data, 0.79, atol=1e-5)
assert np.allclose(optimizer.momentum_buffers[id(theta)], 2.1, atol=1e-5)

theta.grad = np.array(1.0, dtype=np.float32)
optimizer.step()
effective_gradient = 1.0 + 0.1 * 0.79
expected_velocity = 0.9 * 2.1 + effective_gradient
expected_theta = 0.79 - 0.1 * expected_velocity
print("after step 2:", theta.data, "(expected", expected_theta, ")")
assert np.allclose(optimizer.momentum_buffers[id(theta)], expected_velocity, atol=1e-5)
assert np.allclose(theta.data, expected_theta, atol=1e-5)
print("Momentum + weight decay matches the hand-computed two-step trace")

after step 1: 0.79
after step 2: 0.49310002 (expected 0.4931 )
Momentum + weight decay matches the hand-computed two-step trace


### Test 5 вЂ” zero gradient versus grad=None

With `weight_decay=0`: `grad=None` gives no update, a zero gradient gives no *numerical* change (but still runs optimizer logic, e.g. allocates the momentum buffer).

With `weight_decay>0`: `grad=None` still skips the parameter completely, while a zero gradient must shrink the parameter because weight decay is added to it.

In [11]:
# weight_decay = 0
theta = Tensor_CP(np.array([2.0, -3.0], dtype=np.float32), requires_grad=True)
optimizer = SGD([theta], lr=0.1, momentum=0.9)

theta.grad = None
optimizer.step()
assert np.array_equal(theta.data, [2.0, -3.0])
assert id(theta) not in optimizer.momentum_buffers, "grad=None must not allocate a buffer"

theta.grad = np.zeros(2, dtype=np.float32)
optimizer.step()
assert np.array_equal(theta.data, [2.0, -3.0]), "zero gradient: no numerical change"
assert id(theta) in optimizer.momentum_buffers, "zero gradient still runs optimizer logic"
print("weight_decay=0: grad=None skips, zero gradient participates without changing data")

# weight_decay > 0
theta = Tensor_CP(np.array([2.0, -3.0], dtype=np.float32), requires_grad=True)
optimizer = SGD([theta], lr=0.1, weight_decay=0.1)

theta.grad = None
optimizer.step()
assert np.array_equal(theta.data, [2.0, -3.0]), "grad=None skips even with weight decay"

theta.grad = np.zeros(2, dtype=np.float32)
optimizer.step()
expected = np.array([2.0, -3.0], dtype=np.float32) * (1.0 - 0.1 * 0.1)
print("after zero-gradient step with weight decay:", theta.data)
assert np.allclose(theta.data, expected, atol=ATOL, rtol=RTOL), "weight decay must shrink the parameter"
print("weight_decay>0: zero gradient shrinks the parameter, grad=None does not")

weight_decay=0: grad=None skips, zero gradient participates without changing data
after zero-gradient step with weight decay: [ 1.98 -2.97]
weight_decay>0: zero gradient shrinks the parameter, grad=None does not


### Test 6 вЂ” buffer independence

Two parameters of different shapes must get their own momentum buffers with matching shapes, and updating one parameter must not touch the other's buffer.

In [12]:
first = Tensor_CP(np.ones((3,), dtype=np.float32), requires_grad=True)
second = Tensor_CP(np.ones((2, 2), dtype=np.float32), requires_grad=True)
optimizer = SGD([first, second], lr=0.01, momentum=0.9)

for _ in range(3):
    first.grad = np.full((3,), 0.5, dtype=np.float32)
    second.grad = np.full((2, 2), -0.25, dtype=np.float32)
    optimizer.step()

buffer_first = optimizer.momentum_buffers[id(first)]
buffer_second = optimizer.momentum_buffers[id(second)]
assert buffer_first.shape == (3,)
assert buffer_second.shape == (2, 2)
assert not np.shares_memory(buffer_first, buffer_second)

second_snapshot = optimizer.momentum_buffers[id(second)].copy()
first.grad = np.full((3,), 1.0, dtype=np.float32)
second.grad = None
optimizer.step()
assert np.array_equal(optimizer.momentum_buffers[id(second)], second_snapshot), \
    "updating one parameter must not touch the other buffer"
print("Each parameter keeps its own correctly shaped, independent momentum buffer")

Each parameter keeps its own correctly shaped, independent momentum buffer

### Test 7 вЂ” PyTorch parity

TinyTorch and PyTorch run from identical initial parameters with manually assigned, per-step-changing gradients. PyTorch is configured with `dampening=0`, `nesterov=False`, `maximize=False`, and the ordinary loop implementation (`foreach=False`, `fused=False`) for an algorithm-level comparison. Tolerance: `atol=1e-6`, `rtol=1e-5`.

In [13]:
def run_sgd_parity(config, steps, shape):
    rng = np.random.default_rng(0)
    theta0 = np.asarray(rng.normal(size=shape), dtype=np.float32)
    gradients = [np.asarray(rng.normal(size=shape), dtype=np.float32) for _ in range(steps)]

    parameter = Tensor_CP(theta0.copy(), requires_grad=True)
    tiny_optimizer = SGD([parameter], **config)

    torch_parameter = torch.from_numpy(theta0.copy()).requires_grad_(True)
    torch_optimizer = torch.optim.SGD(
        [torch_parameter], lr=config["lr"], momentum=config["momentum"],
        weight_decay=config["weight_decay"], dampening=0.0,
        nesterov=False, maximize=False, foreach=False, fused=False,
    )

    for gradient in gradients:
        parameter.grad = gradient.copy()
        torch_parameter.grad = torch.from_numpy(gradient.copy())
        tiny_optimizer.step()
        torch_optimizer.step()

    reference = torch_parameter.detach().numpy()
    difference = float(np.max(np.abs(parameter.data - reference)))
    assert np.allclose(parameter.data, reference, atol=ATOL, rtol=RTOL), (
        f"SGD parity failed for {config}, {steps} steps, shape {shape}: "
        f"max diff {difference:.2e}"
    )
    return difference


sgd_configs = [
    dict(lr=0.1, momentum=0.0, weight_decay=0.0),
    dict(lr=0.1, momentum=0.0, weight_decay=0.01),
    dict(lr=0.1, momentum=0.9, weight_decay=0.0),
    dict(lr=0.05, momentum=0.9, weight_decay=0.01),
]

print(f"{'lr':>5} {'momentum':>9} {'weight_decay':>13} {'steps':>6} {'max diff':>10}")
for config in sgd_configs:
    for steps in (1, 2, 20):
        difference = max(run_sgd_parity(config, steps, shape) for shape in ((5,), (3, 4)))
        print(f"{config['lr']:>5} {config['momentum']:>9} {config['weight_decay']:>13} "
              f"{steps:>6} {difference:>10.2e}")

print("\nSGD matches torch.optim.SGD for all configurations and step counts")

   lr  momentum  weight_decay  steps   max diff


  0.1       0.0           0.0      1   0.00e+00
  0.1       0.0           0.0      2   0.00e+00
  0.1       0.0           0.0     20   0.00e+00
  0.1       0.0          0.01      1   0.00e+00
  0.1       0.0          0.01      2   0.00e+00
  0.1       0.0          0.01     20   0.00e+00
  0.1       0.9           0.0      1   0.00e+00
  0.1       0.9           0.0      2   0.00e+00
  0.1       0.9           0.0     20   0.00e+00
 0.05       0.9          0.01      1   0.00e+00
 0.05       0.9          0.01      2   0.00e+00
 0.05       0.9          0.01     20   0.00e+00

SGD matches torch.optim.SGD for all configurations and step counts


## 3. Adam tests

### Test 1 вЂ” first step: moments, bias correction, update

theta_0 = 1, g = 0.5, lr = 0.001, beta1 = 0.9, beta2 = 0.999, eps = 1e-8.

- m_1 = 0.1 * 0.5 = 0.05, v_1 = 0.001 * 0.25 = 0.00025
- Bias-corrected: m_hat = 0.05 / (1 - 0.9) = 0.5, v_hat = 0.00025 / (1 - 0.999) = 0.25
- Update: theta_1 = 1 - 0.001 * 0.5 / (sqrt(0.25) + 1e-8) в‰€ 0.999

In [14]:
theta = Tensor_CP(np.array(1.0, dtype=np.float32), requires_grad=True)
optimizer = Adam([theta], lr=0.001)

theta.grad = np.array(0.5, dtype=np.float32)
optimizer.step()

first_moment = optimizer.first_moment_buffers[id(theta)]
second_moment = optimizer.second_moment_buffers[id(theta)]
expected_theta = 1.0 - 0.001 * 0.5 / (np.sqrt(0.25) + 1e-8)

print("first moment:", first_moment)
print("second moment:", second_moment)
print("theta:", theta.data, "(expected", expected_theta, ")")

assert np.allclose(first_moment, 0.05, atol=ATOL)
assert np.allclose(second_moment, 0.00025, atol=1e-8)
assert np.allclose(theta.data, expected_theta, atol=ATOL)
assert optimizer.step_count == 1
assert np.array_equal(theta.grad, np.array(0.5, dtype=np.float32)), ".grad must stay untouched"
print("First Adam step matches the hand computation")

first moment: 0.05
second moment: 0.00025
theta: 0.999 (expected 0.99900000002 )
First Adam step matches the hand computation


### Test 2 вЂ” second step with the same gradient

Feeding g = 0.5 again tests buffer persistence and bias correction:

- m_2 = 0.9 * 0.05 + 0.1 * 0.5 = 0.095, v_2 = 0.999 * 0.00025 + 0.001 * 0.25 = 0.00049975
- After correction both return to m_hat = 0.5 and v_hat = 0.25, so the update size stays в‰€ 0.001 and theta_2 в‰€ 0.998.

In [15]:
theta.grad = np.array(0.5, dtype=np.float32)
optimizer.step()

first_moment = optimizer.first_moment_buffers[id(theta)]
second_moment = optimizer.second_moment_buffers[id(theta)]
expected_theta = float(expected_theta) - 0.001 * 0.5 / (np.sqrt(0.25) + 1e-8)

print("first moment:", first_moment)
print("second moment:", second_moment)
print("theta:", theta.data, "(expected", expected_theta, ")")

assert np.allclose(first_moment, 0.095, atol=ATOL)
assert np.allclose(second_moment, 0.00049975, atol=1e-8)
assert np.allclose(theta.data, expected_theta, atol=ATOL)
assert optimizer.step_count == 2
print("Buffers persisted across steps and bias correction used t=2")

first moment: 0.095
second moment: 0.00049975
theta: 0.998 (expected 0.99800000004 )
Buffers persisted across steps and bias correction used t=2


### Test 3 вЂ” negative gradient moves the parameter up

With g = -0.5 the update direction is negative, so subtracting it must increase theta.

In [16]:
theta = Tensor_CP(np.array(1.0, dtype=np.float32), requires_grad=True)
optimizer = Adam([theta], lr=0.001)

theta.grad = np.array(-0.5, dtype=np.float32)
optimizer.step()

print("theta:", theta.data)
assert theta.data > 1.0, "negative gradient must increase the parameter"
print("Negative gradient increases the parameter")

theta: 1.001
Negative gradient increases the parameter


### Test 4 вЂ” zero gradient (not grad=None)

A fresh optimizer fed a zero gradient produces exactly no update (m = v = 0). With existing state, a zero gradient decays both moments by their beta factors. Neither case may produce NaNs or infinities.

In [17]:
# fresh optimizer: zero gradient -> no meaningful update
theta = Tensor_CP(np.array(1.0, dtype=np.float32), requires_grad=True)
optimizer = Adam([theta], lr=0.001)
theta.grad = np.array(0.0, dtype=np.float32)
optimizer.step()
assert np.allclose(theta.data, 1.0, atol=1e-8)
assert np.all(np.isfinite(theta.data))
assert np.allclose(optimizer.first_moment_buffers[id(theta)], 0.0)
assert np.allclose(optimizer.second_moment_buffers[id(theta)], 0.0)
print("Fresh optimizer + zero gradient: parameter unchanged, buffers stay zero")

# existing state: zero gradient decays the moments
theta = Tensor_CP(np.array(1.0, dtype=np.float32), requires_grad=True)
optimizer = Adam([theta], lr=0.001)
theta.grad = np.array(0.5, dtype=np.float32)
optimizer.step()
theta.grad = np.array(0.0, dtype=np.float32)
optimizer.step()

first_moment = optimizer.first_moment_buffers[id(theta)]
second_moment = optimizer.second_moment_buffers[id(theta)]
assert np.allclose(first_moment, 0.9 * 0.05, atol=ATOL), "first moment must decay by beta1"
assert np.allclose(second_moment, 0.999 * 0.00025, atol=1e-8), "second moment must decay by beta2"
assert np.all(np.isfinite(theta.data))
print("Existing state + zero gradient: moments decay by beta1/beta2, no NaN or infinity")

Fresh optimizer + zero gradient: parameter unchanged, buffers stay zero
Existing state + zero gradient: moments decay by beta1/beta2, no NaN or infinity


### Test 5 вЂ” classical (coupled) Adam weight decay

This implementation uses coupled weight decay: g' = g + lambda * theta. With theta = 1, g = 0, lambda = 0.1 the effective gradient becomes 0.1, so the parameter must move even though the loss gradient is zero.

**Do not** compare this configuration against `torch.optim.AdamW` вЂ” AdamW decouples weight decay from the moment estimates and produces different numbers. The parity partner is `torch.optim.Adam(weight_decay=...)`, which uses the same coupled formulation.

In [18]:
theta = Tensor_CP(np.array(1.0, dtype=np.float32), requires_grad=True)
optimizer = Adam([theta], lr=0.001, weight_decay=0.1)

theta.grad = np.array(0.0, dtype=np.float32)
optimizer.step()

# effective gradient 0.1: m_hat = 0.1, v_hat = 0.01 -> update = lr * 0.1 / (0.1 + eps)
expected_theta = 1.0 - 0.001 * 0.1 / (np.sqrt(0.01) + 1e-8)
print("theta:", theta.data, "(expected", expected_theta, ")")
assert theta.data < 1.0, "weight decay must move the parameter despite a zero loss gradient"
assert np.allclose(theta.data, expected_theta, atol=ATOL)
print("Coupled weight decay updates the parameter with a zero loss gradient")

theta: 0.999 (expected 0.9990000001 )
Coupled weight decay updates the parameter with a zero loss gradient


### Test 6 вЂ” epsilon stability

Zero gradients, extremely small gradients (1e-12), and parameters near zero must never produce division by zero, NaN, or infinity.

In [19]:
stability_cases = [
    ("zero gradients", np.zeros(4, dtype=np.float32),
     np.array([1.0, -1.0, 0.5, 2.0], dtype=np.float32)),
    ("tiny gradients (1e-12)", np.full(4, 1e-12, dtype=np.float32),
     np.array([1.0, -1.0, 0.5, 2.0], dtype=np.float32)),
    ("parameters near zero", np.full(4, 1e-12, dtype=np.float32),
     np.full(4, 1e-20, dtype=np.float32)),
]

for label, gradient, initial in stability_cases:
    theta = Tensor_CP(initial.copy(), requires_grad=True)
    optimizer = Adam([theta], lr=0.001)
    for _ in range(10):
        theta.grad = gradient.copy()
        optimizer.step()
    assert np.all(np.isfinite(theta.data)), label
    assert np.all(np.isfinite(optimizer.first_moment_buffers[id(theta)])), label
    assert np.all(np.isfinite(optimizer.second_moment_buffers[id(theta)])), label
    print(f"OK   {label}: finite after 10 steps")

print("\nEpsilon keeps every case free of NaN, infinity, and division by zero")

OK   zero gradients: finite after 10 steps
OK   tiny gradients (1e-12): finite after 10 steps
OK   parameters near zero: finite after 10 steps

Epsilon keeps every case free of NaN, infinity, and division by zero


### Test 7 вЂ” moment-buffer independence

Each parameter needs its own first- and second-moment arrays with the right shape and float32 dtype, sharing no memory with any other buffer.

In [20]:
first = Tensor_CP(np.ones((3,), dtype=np.float32), requires_grad=True)
second = Tensor_CP(np.ones((2, 2), dtype=np.float32), requires_grad=True)
optimizer = Adam([first, second], lr=0.001)

for _ in range(3):
    first.grad = np.full((3,), 0.5, dtype=np.float32)
    second.grad = np.full((2, 2), -0.25, dtype=np.float32)
    optimizer.step()

buffers = {
    "first moment (3,)": optimizer.first_moment_buffers[id(first)],
    "first moment (2,2)": optimizer.first_moment_buffers[id(second)],
    "second moment (3,)": optimizer.second_moment_buffers[id(first)],
    "second moment (2,2)": optimizer.second_moment_buffers[id(second)],
}

assert buffers["first moment (3,)"].shape == (3,)
assert buffers["first moment (2,2)"].shape == (2, 2)
assert buffers["second moment (3,)"].shape == (3,)
assert buffers["second moment (2,2)"].shape == (2, 2)

names = list(buffers)
for index, name in enumerate(names):
    assert buffers[name].dtype == np.float32, f"{name} must be float32"
    for other in names[index + 1:]:
        assert not np.shares_memory(buffers[name], buffers[other]), \
            f"{name} and {other} share memory"

print("All four moment buffers are separate float32 arrays with the correct shapes")

All four moment buffers are separate float32 arrays with the correct shapes


### Test 8 вЂ” PyTorch parity

Compared against `torch.optim.Adam` with matching `lr`, `betas`, `eps`, and `weight_decay`, using the ordinary loop implementation (`foreach=False`, `fused=False`). Scalar, vector, and matrix parameters run together in one optimizer with per-step-changing gradients. Tolerance `atol=1e-6` up to 20 steps; the 100-step run uses `atol=1e-5` for accumulated float32 rounding.

In [21]:
def run_adam_parity(config, steps, shapes, atol=ATOL):
    rng = np.random.default_rng(1)
    initial = [np.asarray(rng.normal(size=shape), dtype=np.float32) for shape in shapes]

    tiny_params = [Tensor_CP(array.copy(), requires_grad=True) for array in initial]
    torch_params = [torch.from_numpy(array.copy()).requires_grad_(True) for array in initial]

    tiny_optimizer = Adam(
        tiny_params, lr=config["lr"], betas=(config["beta1"], config["beta2"]),
        epsilon=config["epsilon"], weight_decay=config["weight_decay"],
    )
    torch_optimizer = torch.optim.Adam(
        torch_params, lr=config["lr"], betas=(config["beta1"], config["beta2"]),
        eps=config["epsilon"], weight_decay=config["weight_decay"],
        foreach=False, fused=False,
    )

    for _ in range(steps):
        for tiny_parameter, torch_parameter in zip(tiny_params, torch_params):
            gradient = np.asarray(rng.normal(size=tiny_parameter.shape), dtype=np.float32)
            tiny_parameter.grad = gradient.copy()
            torch_parameter.grad = torch.from_numpy(gradient.copy())
        tiny_optimizer.step()
        torch_optimizer.step()

    max_difference = 0.0
    for tiny_parameter, torch_parameter in zip(tiny_params, torch_params):
        reference = torch_parameter.detach().numpy()
        max_difference = max(max_difference,
                             float(np.max(np.abs(tiny_parameter.data - reference))))
        assert np.allclose(tiny_parameter.data, reference, atol=atol, rtol=RTOL), (
            f"Adam parity failed for {config}, {steps} steps: max diff {max_difference:.2e}"
        )
    return max_difference


adam_configs = [
    dict(lr=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8, weight_decay=0.0),
    dict(lr=0.01, beta1=0.8, beta2=0.99, epsilon=1e-8, weight_decay=0.0),
    dict(lr=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8, weight_decay=0.01),
]
shapes = ((), (5,), (3, 4))

print(f"{'lr':>6} {'beta1':>6} {'beta2':>6} {'weight_decay':>13} {'steps':>6} {'max diff':>10}")
for config in adam_configs:
    for steps in (1, 2, 20):
        difference = run_adam_parity(config, steps, shapes)
        print(f"{config['lr']:>6} {config['beta1']:>6} {config['beta2']:>6} "
              f"{config['weight_decay']:>13} {steps:>6} {difference:>10.2e}")

long_run_difference = run_adam_parity(adam_configs[0], 100, ((5,), (3, 4)), atol=1e-5)
print(f"\n100-step run (atol=1e-5): max diff {long_run_difference:.2e}")

# two repeated-gradient steps against PyTorch
tiny_theta = Tensor_CP(np.array(1.0, dtype=np.float32), requires_grad=True)
torch_theta = torch.tensor(1.0, requires_grad=True)
tiny_optimizer = Adam([tiny_theta], lr=0.001)
torch_optimizer = torch.optim.Adam([torch_theta], lr=0.001, foreach=False, fused=False)
for _ in range(2):
    tiny_theta.grad = np.array(0.5, dtype=np.float32)
    torch_theta.grad = torch.tensor(0.5)
    tiny_optimizer.step()
    torch_optimizer.step()
assert np.allclose(tiny_theta.data, torch_theta.detach().numpy(), atol=ATOL, rtol=RTOL)
print("Repeated-gradient two-step run matches PyTorch")

print("\nAdam matches torch.optim.Adam for all configurations")

    lr  beta1  beta2  weight_decay  steps   max diff
 0.001    0.9  0.999           0.0      1   0.00e+00
 0.001    0.9  0.999           0.0      2   0.00e+00
 0.001    0.9  0.999           0.0     20   3.73e-09
  0.01    0.8   0.99           0.0      1   1.86e-09
  0.01    0.8   0.99           0.0      2   9.31e-10
  0.01    0.8   0.99           0.0     20   2.98e-08
 0.001    0.9  0.999          0.01      1   0.00e+00
 0.001    0.9  0.999          0.01      2   0.00e+00
 0.001    0.9  0.999          0.01     20   3.73e-09



100-step run (atol=1e-5): max diff 1.49e-08
Repeated-gradient two-step run matches PyTorch

Adam matches torch.optim.Adam for all configurations


### Intermittent grad=None вЂ” known limitation

Parameter A receives a gradient every step; parameter B stays `grad=None` for three steps and gets its first gradient at step 4.

Adam keeps a single global `self.step_count`, so B's first update is bias-corrected as though it were B's *fourth* update: TinyTorch gives 0.99941885 where PyTorch gives 0.999 (PyTorch stores a per-parameter counter in `state[param]["step"]`). Per the final pass criteria this is **documented as a known limitation**; the fix would be a per-parameter step counter keyed by tensor identity. The last assert flips if the limitation is ever fixed, flagging this cell for an upgrade to a parity test.

In [22]:
tiny_a = Tensor_CP(np.array(1.0, dtype=np.float32), requires_grad=True)
tiny_b = Tensor_CP(np.array(1.0, dtype=np.float32), requires_grad=True)
tiny_optimizer = Adam([tiny_a, tiny_b], lr=0.001)

torch_a = torch.tensor(1.0, requires_grad=True)
torch_b = torch.tensor(1.0, requires_grad=True)
torch_optimizer = torch.optim.Adam([torch_a, torch_b], lr=0.001, foreach=False, fused=False)

for step_index in range(1, 5):
    tiny_a.grad = np.array(0.5, dtype=np.float32)
    torch_a.grad = torch.tensor(0.5)
    if step_index >= 4:
        tiny_b.grad = np.array(0.5, dtype=np.float32)
        torch_b.grad = torch.tensor(0.5)
    else:
        tiny_b.grad = None
        torch_b.grad = None
    tiny_optimizer.step()
    torch_optimizer.step()

print("A: tinytorch", tiny_a.data, "pytorch", torch_a.detach().numpy())
print("B: tinytorch", tiny_b.data, "pytorch", torch_b.detach().numpy())

assert np.allclose(tiny_a.data, torch_a.detach().numpy(), atol=ATOL, rtol=RTOL), \
    "A receives gradients every step and must still match PyTorch"
b_matches = np.allclose(tiny_b.data, torch_b.detach().numpy(), atol=ATOL, rtol=RTOL)
assert not b_matches, ("the known limitation no longer reproduces - Adam now handles "
                       "intermittent gradients; upgrade this cell to assert parity")
print("Known limitation reproduced: the global step_count over-corrects the late-starting parameter")

A: tinytorch 0.99600005 pytorch 0.99600005
B: tinytorch 0.99941885 pytorch 0.999
Known limitation reproduced: the global step_count over-corrects the late-starting parameter


## 4. End-to-end training tests

### Quadratic minimization

Minimize L = (x - 3)^2 from x = 0 with the full loop: build the loss from `Tensor_CP` operations, `loss.backward()`, `optimizer.step()`, `optimizer.zero_grad()`. The gradient check inside the loop proves no stale gradients accumulate: before each `backward()` the fresh analytic gradient 2(x - 3) must match exactly.

In [23]:
def minimize_quadratic(optimizer_factory, n_steps, check_gradient=False):
    x = Tensor_CP(np.array(0.0, dtype=np.float32), requires_grad=True)
    optimizer = optimizer_factory(x)
    losses = []
    for _ in range(n_steps):
        difference = x - 3.0
        loss = difference * difference
        expected_gradient = 2.0 * (float(x.data) - 3.0)
        loss.backward()
        if check_gradient:
            assert np.allclose(x.grad, expected_gradient, atol=1e-4), \
                "stale gradients accumulated across iterations"
        optimizer.step()
        optimizer.zero_grad()
        losses.append(float(loss.data))
    assert x.grad is None, "zero_grad() must leave the gradient reset"
    return x, losses


x_sgd, losses_sgd = minimize_quadratic(lambda p: SGD([p], lr=0.1), 100, check_gradient=True)
x_adam, losses_adam = minimize_quadratic(lambda p: Adam([p], lr=0.2), 500, check_gradient=True)

print(f"SGD : x = {float(x_sgd.data):.6f}, first loss = {losses_sgd[0]:.4f}, "
      f"last loss = {losses_sgd[-1]:.2e}")
print(f"Adam: x = {float(x_adam.data):.6f}, first loss = {losses_adam[0]:.4f}, "
      f"last loss = {losses_adam[-1]:.2e}")

assert losses_sgd[-1] < losses_sgd[0]
assert losses_adam[-1] < losses_adam[0]
assert abs(float(x_sgd.data) - 3.0) < 1e-3
assert abs(float(x_adam.data) - 3.0) < 0.05
print("Both optimizers drive x to 3 with decreasing loss and no gradient accumulation")

SGD : x = 3.000000, first loss = 9.0000, last loss = 2.27e-13
Adam: x = 3.000000, first loss = 9.0000, last loss = 0.00e+00
Both optimizers drive x to 3 with decreasing loss and no gradient accumulation


### Small linear regression

Fit y = 2x + 1 with two scalar parameters through the full chain: tensor operations в†’ autograd в†’ `MSELoss_CP` в†’ optimizer в†’ parameter update. Both optimizers must drive w toward 2 and b toward 1 with a steadily decreasing loss.

In [24]:
regression_rng = np.random.default_rng(3)
x_values = np.asarray(regression_rng.uniform(-1.0, 1.0, size=64), dtype=np.float32)
y_values = (2.0 * x_values + 1.0).astype(np.float32)


def train_linear_regression(optimizer_factory, n_steps):
    weight = Tensor_CP(np.array(0.0, dtype=np.float32), requires_grad=True)
    bias = Tensor_CP(np.array(0.0, dtype=np.float32), requires_grad=True)
    optimizer = optimizer_factory([weight, bias])
    loss_function = MSELoss_CP()
    inputs = Tensor_CP(x_values)
    targets = Tensor_CP(y_values)

    losses = []
    for _ in range(n_steps):
        predictions = weight * inputs + bias
        loss = loss_function(predictions, targets)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        losses.append(float(loss.data))
    assert weight.grad is None and bias.grad is None, "gradients must reset each iteration"
    return weight, bias, losses


for label, factory, n_steps in (
    ("SGD", lambda ps: SGD(ps, lr=0.3), 300),
    ("Adam", lambda ps: Adam(ps, lr=0.05), 1000),
):
    weight, bias, losses = train_linear_regression(factory, n_steps)
    print(f"{label:>4}: w = {float(weight.data):.4f}, b = {float(bias.data):.4f}, "
          f"first loss = {losses[0]:.4f}, last loss = {losses[-1]:.2e}")
    assert losses[len(losses) // 2] < losses[0], f"{label}: loss must decrease steadily"
    assert losses[-1] < 1e-4, f"{label}: final loss must be near zero"
    assert abs(float(weight.data) - 2.0) < 0.05, f"{label}: w must approach 2"
    assert abs(float(bias.data) - 1.0) < 0.05, f"{label}: b must approach 1"

print("\nBoth optimizers recover w = 2, b = 1 through the full autograd chain")

 SGD: w = 2.0000, b = 1.0000, first loss = 2.1865, last loss = 2.43e-14


Adam: w = 2.0000, b = 1.0000, first loss = 2.1865, last loss = 0.00e+00

Both optimizers recover w = 2, b = 1 through the full autograd chain


## 5. Performance

Benchmarked separately from correctness. Let P be the total number of parameter elements:

| Optimizer | Time per step | Persistent optimizer state |
|---|---|---|
| SGD | O(P) | ~0 |
| SGD + momentum | O(P) | one parameter-sized buffer |
| Adam | O(P) | two parameter-sized buffers |

For float32 and one million elements: parameter в‰€ 4 MB, SGD momentum state в‰€ 4 MB, Adam moment state в‰€ 8 MB (excluding gradients, temporaries, and Python-object overhead).

Layouts: one tensor with 1k / 100k / 1M elements, plus 1,000 small tensors totaling 1M elements вЂ” the last case exposes the per-tensor Python-loop overhead. Gradients are pre-created outside the timed region; 5 warm-up steps allocate state buffers before timing. The "growth" column is the RSS change across the timed steps вЂ” it should stay near zero once buffers exist. The max-diff column compares final parameters against PyTorch running the identical schedule.

In [25]:
process = psutil.Process()

layouts = [
    ("1 tensor x 1k", [(1_000,)], 1_000),
    ("1 tensor x 100k", [(100_000,)], 500),
    ("1 tensor x 1M", [(1_000_000,)], 200),
    ("1000 tensors x 1k", [(1_000,)] * 1_000, 200),
]

optimizer_setups = [
    ("SGD",
     lambda ps: SGD(ps, lr=0.01),
     lambda ps: torch.optim.SGD(ps, lr=0.01, foreach=False, fused=False)),
    ("SGD+momentum",
     lambda ps: SGD(ps, lr=0.01, momentum=0.9),
     lambda ps: torch.optim.SGD(ps, lr=0.01, momentum=0.9, foreach=False, fused=False)),
    ("Adam",
     lambda ps: Adam(ps, lr=0.001),
     lambda ps: torch.optim.Adam(ps, lr=0.001, foreach=False, fused=False)),
]

WARMUP_STEPS = 5


def benchmark_layout(tiny_factory, torch_factory, shapes, n_steps):
    rng = np.random.default_rng(11)
    arrays = [np.asarray(rng.normal(size=shape), dtype=np.float32) for shape in shapes]
    gradients = [np.asarray(rng.normal(size=shape), dtype=np.float32) for shape in shapes]

    parameters = [Tensor_CP(array.copy(), requires_grad=True) for array in arrays]
    for parameter, gradient in zip(parameters, gradients):
        parameter.grad = gradient
    optimizer = tiny_factory(parameters)

    memory_start_mb = process.memory_info().rss / 2**20
    for _ in range(WARMUP_STEPS):
        optimizer.step()
    memory_after_warmup_mb = process.memory_info().rss / 2**20

    samples_ms = []
    for _ in range(n_steps):
        start = perf_counter()
        optimizer.step()
        samples_ms.append((perf_counter() - start) * 1_000)
    memory_after_run_mb = process.memory_info().rss / 2**20

    # PyTorch reference over the identical schedule for the numerical difference
    torch_parameters = [torch.from_numpy(array.copy()).requires_grad_(True) for array in arrays]
    for torch_parameter, gradient in zip(torch_parameters, gradients):
        torch_parameter.grad = torch.from_numpy(gradient.copy())
    torch_optimizer = torch_factory(torch_parameters)
    for _ in range(WARMUP_STEPS + n_steps):
        torch_optimizer.step()

    max_difference = max(
        float(np.max(np.abs(parameter.data - torch_parameter.detach().numpy())))
        for parameter, torch_parameter in zip(parameters, torch_parameters)
    )
    return dict(
        mean_ms=mean(samples_ms),
        median_ms=median(samples_ms),
        state_mb=memory_after_warmup_mb - memory_start_mb,
        growth_mb=memory_after_run_mb - memory_after_warmup_mb,
        rss_mb=memory_after_run_mb,
        max_difference=max_difference,
    )


print(f"{'optimizer':<13} {'layout':<18} {'mean ms':>8} {'median ms':>10} "
      f"{'state MB':>9} {'growth MB':>10} {'rss MB':>8} {'max diff':>10}")
for optimizer_name, tiny_factory, torch_factory in optimizer_setups:
    for layout_name, shapes, n_steps in layouts:
        stats = benchmark_layout(tiny_factory, torch_factory, shapes, n_steps)
        print(f"{optimizer_name:<13} {layout_name:<18} {stats['mean_ms']:>8.4f} "
              f"{stats['median_ms']:>10.4f} {stats['state_mb']:>9.1f} "
              f"{stats['growth_mb']:>10.2f} {stats['rss_mb']:>8.0f} "
              f"{stats['max_difference']:>10.2e}")
        assert stats["growth_mb"] < 50, (
            f"{optimizer_name} on {layout_name}: memory grew {stats['growth_mb']:.1f} MB "
            f"across repeated steps - state is being recreated or retained incorrectly"
        )

print("\nMemory is stable across repeated steps for every optimizer and layout")

optimizer     layout              mean ms  median ms  state MB  growth MB   rss MB   max diff
SGD           1 tensor x 1k        0.0021     0.0021       0.0       0.00      333   7.57e-05


SGD           1 tensor x 100k      0.0338     0.0300       0.4       0.00      333   2.28e-04


SGD           1 tensor x 1M        2.7150     2.5659       0.0       0.00      346   6.48e-05


SGD           1000 tensors x 1k    2.5494     2.6022       0.0       0.00      344   6.48e-05
SGD+momentum  1 tensor x 1k        0.0042     0.0030       0.0       0.03      337   1.53e-05


SGD+momentum  1 tensor x 100k      0.1084     0.1030       0.8       0.00      339   1.53e-05


SGD+momentum  1 tensor x 1M        8.4325     6.5734       3.8       0.00      354   1.53e-05


SGD+momentum  1000 tensors x 1k    7.3535     6.9089       4.1       0.00      349   1.53e-05
Adam          1 tensor x 1k        0.0196     0.0162       0.0       0.00      340   5.60e-06


Adam          1 tensor x 100k      0.6552     0.5949       2.2       0.00      344   8.34e-06


Adam          1 tensor x 1M       25.1225    24.5659       7.6       0.00      363   1.07e-05


Adam          1000 tensors x 1k   24.6413    24.2216       9.2       0.00      370   1.07e-05

Memory is stable across repeated steps for every optimizer and layout


### Results

- Every manual formula test passes for SGD (plain, weight decay, momentum, momentum + weight decay) and Adam (both moments, bias correction, repeated-gradient persistence, coupled weight decay, epsilon stability).
- `step()` returns `None`, never mutates `.grad`, skips `grad=None` parameters, and preserves shapes and dtypes; `zero_grad()` resets every gradient to `None`.
- State buffers persist across steps, stay independent per parameter, and keep correct shapes and float32 dtype.
- PyTorch parity holds at `atol=1e-6` for 20+ consecutive updates in every configuration (loop implementation, `foreach=False`, `fused=False`), and at `atol=1e-5` over 100 Adam steps.
- The quadratic and the linear regression both converge through the full tensor в†’ autograd в†’ `MSELoss_CP` в†’ optimizer chain.
- Memory stabilizes after the state buffers are allocated; repeated steps show no growth.
- An empty parameter list is deliberately allowed as a no-op; duplicate parameters (double update per step) and boolean hyperparameters such as `lr=True` are accepted вЂ” both documented as known limitations with pinning tests.
- Adam's intermittent `grad=None` case is a **documented known limitation**: the single global `step_count` bias-corrects a late-starting parameter as though it had taken every previous step (measured: 0.99941885 vs PyTorch's 0.999). A per-parameter step counter keyed by tensor identity would restore parity.
- Performance follows the expected O(P) shape: a few large arrays are cheap, 1,000 small tensors pay the per-tensor Python overhead, and Adam costs more than SGD because it performs more elementwise passes over the data.

Timings depend on the CPU, thread settings, and system load, so rerun the benchmark cells on the target machine rather than treating one run as a universal result.